# imports

In [1]:
import numpy as np
import pandas as pd
import hypertools as hyp
import numpy as np
from scipy.spatial.distance import cdist
from scipy.signal import resample
from scipy import ndimage
from scipy.stats import zscore
from scipy.spatial.distance import correlation
%matplotlib inline

# paths to data dirs

In [2]:
annot_dir = '../../data/annotations_dfs/'

# load annotations dataframes

In [11]:
atlep1_df = pd.read_pickle(annot_dir+'atlep1.p')
atlep2_df = pd.read_pickle(annot_dir+'atlep2.p')
arrdev_df = pd.read_pickle(annot_dir+'arrdev.p')

# model parameters

In [4]:
n_topics = 100
episode_wsize = 50
recall_wsize = 10

# vectorizer parameters
vectorizer_params = {
    'model' : 'CountVectorizer', 
    'params' : {
        'stop_words' : 'english'
    }
}

# topic model parameters
semantic_params = {
    'model' : 'LatentDirichletAllocation', 
    'params' : {
        'n_components' : n_topics,
        'learning_method' : 'batch',
        'random_state' : 0,
    }
}

# fit topic model to episode annotations

In [5]:
def fit_episode_model(episode_df, n_topics, episode_wsize, vec_params, sem_params):
    
    # throw all annotations into bag of words to train model
    episode_bag = episode_df.loc[:,'Narrative details (external events)':'Setting'].apply(lambda x: ', '.join(x.fillna('')), axis=1).values.tolist()
    
    # create list for annotation sliding windows (of size w_size)
    episode_w = []
    for idx, sentence in enumerate(episode_bag):
        episode_w.append(','.join(episode_bag[idx:idx+episode_wsize]))
        
    # use hypertools to create episode model
    return hyp.tools.format_data(episode_w, vectorizer=vec_params, semantic=sem_params, corpus=episode_w)[0]

In [12]:
atlep1_model = fit_episode_model(atlep1_df, n_topics, episode_wsize, vectorizer_params, semantic_params)
atlep2_model = fit_episode_model(atlep2_df, n_topics, episode_wsize, vectorizer_params, semantic_params)
arrdev_model = fit_episode_model(arrdev_df, n_topics, episode_wsize, vectorizer_params, semantic_params)

# save video models